# PRE-stop: luật dừng unlearning dựa trên loss trước khi học forget set

Quy trình mỗi seed: giai đoạn 1 (θ_pre, tối đa 300 epoch, dừng sớm patience 5 theo loss train R1) -> giai đoạn 2 (θ0) -> oracle θ_R2 x3 -> quỹ đạo unlearning (`ga`, `scrub`, `salun`, `tarun`, `ft`, đếm theo epoch, đúng code gốc từng phương pháp; quỹ đạo chạy đến khi mọi luật PRE online đã quyết định, không đặt trần epoch, `tarun` đúng 2 epoch).
Cuối cùng: Cổng 1 (chỉ cảnh báo) và phân tích, báo cáo median/IQR, so sánh đủ chỉ số với mô hình retrain.

**Cách chạy:** bật Accelerator = GPU, rồi `Run All`. Notebook chạy tuần tự **từng seed** (pre -> cfg -> traj) và lưu zip sau mỗi seed, nên nếu hết giờ thì phần đã xong vẫn còn. Bước nào đã có kết quả sẽ tự bỏ qua, nên chạy lại là tiếp tục.
Thời gian: lần chạy trước (quỹ đạo cố định 10 epoch) mất khoảng 1,7 đến 2 giờ mỗi seed (pre 40-47 phút, cfg 26 phút, traj 33 phút). Quỹ đạo dài hơn nên bước traj có thể lên tới gấp ~3 lần (chưa đo, không có trần nên có thể lâu hơn nhiều nếu có luật không bao giờ quyết định; theo dõi log).
Mục 3b (tuỳ chọn, exploratory): lưới lr nhỏ hơn cho `ga` và `scrub`, báo cáo riêng ở `results_lrgrid/`.

In [ ]:
import os, sys, json, glob, shutil, time, subprocess

REPO_URL = "https://github.com/tiensinh2/Stopping_for_MU.git"
REPO_DIR = "/kaggle/working/ralm_repo"
OUT      = "/kaggle/working/pre_out"       # đổi tên nếu muốn chạy mới hoàn toàn (cache θ_pre cũ khác cấu hình sẽ bị từ chối)
PROFILE  = "kaggle"
SEEDS    = [0]                             # chỉ 1 seed (theo yêu cầu); muốn thêm seed thì [0, 1] hoặc chạy lại với seed khác và cùng OUT
FTYPES   = "random-450 random-1350 class-0 hard-450"
METHODS  = "ga scrub salun tarun ft"       # tarun chỉ chạy class-0, ft chỉ chạy random-450 (code gốc)
RUN_TESTS = True                           # pytest chạy trong tiến trình riêng, lỗi không làm dừng notebook
RESTORE_FROM = ""                          # vd "/kaggle/input/<ten>/pre_out" (chỉ dùng kết quả của CÙNG cấu hình giai đoạn 1 mới này)

In [ ]:
%cd /kaggle/working
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://%s@" % UserSecretsClient().get_secret("GH_TOKEN"))
except Exception:
    pass
if os.path.isdir(REPO_DIR + "/.git"):
    !cd {REPO_DIR} && git pull {url} main
else:
    !git clone {url} {REPO_DIR}
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL}
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q -r requirements.txt
if RESTORE_FROM and os.path.isdir(RESTORE_FROM):
    shutil.copytree(RESTORE_FROM, OUT, dirs_exist_ok=True)
    print("đã khôi phục", RESTORE_FROM, "->", OUT)

import torch
print("GPU:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Không thấy GPU: bật Accelerator = GPU rồi Restart session"


def run(cmd, check=True):
    """Chạy lệnh trong tiến trình riêng, in log ngay khi có; trả về mã thoát."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise SystemExit(f"Lệnh lỗi (mã {rc}): {cmd}")
    return rc


def cli(cmd, seeds, extra=""):
    return f"python -m pre_stop.run {cmd} --profile {PROFILE} --out {OUT} --seeds {' '.join(map(str, seeds))} {extra}"


def status():
    for s in SEEDS:
        has = lambda p: "có" if os.path.exists(p) else "chưa"
        n_cfg = len(glob.glob(f"{OUT}/ckpt/theta0/*_seed{s}.pth"))
        n_traj = len(glob.glob(f"{OUT}/logs/traj_*_seed{s}_u0_meta.json"))
        print(f"seed {s}: θ_pre {has(f'{OUT}/ckpt/pre/seed{s}.pth')} | cấu hình giai đoạn 2: {n_cfg}/4 | quỹ đạo: {n_traj}/14")


status()

## 0. Kiểm thử (không bắt buộc)
Chạy trong tiến trình riêng. Nếu pytest crash (vd segfault do môi trường) thì chỉ in cảnh báo, các bước thật không đi qua đường đó.

In [ ]:
if RUN_TESTS:
    rc = run("python -m pytest tests/test_pre_stop.py -x -q", check=False)
    print("pytest mã thoát:", rc, "(0 = qua; khác 0: gửi log cho người phụ trách, vẫn có thể chạy tiếp)")

## 1. Chạy từng seed: θ_pre -> θ0 + oracle -> quỹ đạo
- `pre` dừng với `SystemExit` nếu test acc của θ_pre < 90% (cổng mục 5): khi đó gửi log, không tự đổi công thức.
- Cổng 1 (e_pre/ν > 10) chỉ cảnh báo ở bước sau, không chặn.
- Sau mỗi seed notebook lưu `pre_out_light.zip` (không kèm checkpoint nặng) vào `/kaggle/working`.

In [ ]:
timing = {}
for s in SEEDS:
    t0 = time.time()
    run(cli("pre",  [s]))
    t1 = time.time()
    run(cli("cfg",  [s], f"--ftypes {FTYPES}"))
    t2 = time.time()
    run(cli("traj", [s], f"--ftypes {FTYPES} --methods {METHODS}"))
    t3 = time.time()
    timing[s] = {"pre_min": (t1 - t0) / 60, "cfg_min": (t2 - t1) / 60, "traj_min": (t3 - t2) / 60}
    print(f"=== seed {s} xong: " + ", ".join(f"{k} {v:.0f}" for k, v in timing[s].items()) + " (phút) ===", flush=True)
    run(f'cd /kaggle/working && rm -f pre_out_light.zip && zip -qr pre_out_light.zip {os.path.basename(OUT)} -x "{os.path.basename(OUT)}/ckpt/*"', check=False)
status()

## 2. Giai đoạn 1 đã dừng ở đâu, và Cổng 1

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
rows = []
for s in SEEDS:
    st = json.load(open(f"{OUT}/data/pre_stats_seed{s}.json", encoding="utf-8"))
    r = st.get("stage1_run", {})
    rows.append({"seed": s, "A_T_pre": round(st["A_T_pre"], 4), "epochs_run": r.get("epochs_run"),
                 "best_epoch": r.get("best_epoch"), "best_loss": r.get("best_loss"),
                 "stopped_early": r.get("stopped_early"), "gate_passed": st["gate_passed"]})
print(pd.DataFrame(rows).to_string(index=False))

run(cli("gate1", SEEDS, f"--ftypes {FTYPES}"), check=False)      # chỉ cảnh báo
g = pd.read_csv(OUT + "/results/gate1.csv")
print(g.round(3).to_string(index=False))

In [ ]:
for m in sorted(glob.glob(OUT + "/logs/traj_*_meta.json")):
    d = json.load(open(m, encoding="utf-8"))
    print(d["run_id"], "epoch", d["n_epochs"], "snapshot", d["n_snapshots"], "updates", d["n_updates"], "dừng:", d.get("stop_reason", "?"), "| luật chưa quyết định:", d.get("n_rules_undecided", "?"), "| diverged", d["diverged"])
print(pd.DataFrame(timing).T.round(1) if timing else "không có số đo thời gian (các bước đã bỏ qua vì có sẵn kết quả)")

## 3. Phân tích
Median và IQR, khoảng cách chuẩn hoá bằng ν. Báo cáo gồm: bảng tiền đề, bảng chính theo luật, **mục 2b so sánh đủ chỉ số với mô hình retrain** (L/A trên F, R, V, test; kèm khoảng cách tới retrain), tiêu chí mục 12 và điều kiện bác bỏ. Kết quả khám phá phải ghi nhãn exploratory.

In [ ]:
run(cli("analyze", SEEDS, f"--ftypes {FTYPES} --methods {METHODS}"))
print(open(OUT + "/results/report_auto.md", encoding="utf-8").read())

## 3b. [EXPLORATORY] Lưới lr nhỏ hơn cho `ga` và `scrub`
Lý do: lr của code gốc (`ga` SGD 0,01; `scrub` Adam 5e-4) được chọn cho mô hình/dữ liệu của bài báo gốc; θ_pre của ta đã ở đáy loss nên có thể quá lớn (`ga` hỏng utility ngay khi F chạm retrain; `scrub` hỏng ngay epoch 1). Mỗi điểm lưới = **hệ số nhân `s` lên lr gốc**, giữ nguyên mọi thứ khác. Với `scrub@s` (s ≠ 1) lr **giữ không đổi**, không giảm ×0,1 ở epoch 5, 8, 9 (lịch gốc làm lr về ~1e-3 lần sau 9 epoch nên quỹ đạo kẹt dao động); `scrub` s = 1 giữ lịch gốc.
- Quỹ đạo vẫn **không có trần epoch**: chạy đến khi mọi luật PRE online quyết định hoặc phân kỳ. Riêng ở ô lưới này có thêm tiêu chí tuỳ chọn `LR_PLATEAU_W` (do tôi đề xuất, không phải trần): dừng khi quỹ đạo tới vùng loss ổn định, không tối ưu thêm (trung bình nửa sau so với nửa trước của `W` epoch gần nhất đổi < 5% ở L_F, L_R, L_V), vì lr rất nhỏ có thể khiến luật không bao giờ quyết định. Đặt `LR_PLATEAU_W = None` để tắt hẳn (có thể chạy rất lâu).
- Báo cáo ghi vào `results_lrgrid/`, **không ghi đè** `results/`. Dòng `ga` và `scrub` (lr gốc, `s = 1`) được dùng lại từ ô trên để đặt cạnh các điểm lưới.
- Đây là quét siêu tham số **khám phá**: lr phải chọn bằng tiêu chí không dùng oracle (L_V không vượt rào G_V, L_F có tiến về P), không chọn theo khoảng cách tới retrain rồi báo như kết quả chính.
- Cần `pre` và `cfg` đã có sẵn cho các loại F được chọn (dùng chung `OUT`).

In [ ]:
LR_SCALES     = [0.5, 0.2, 0.1]           # hệ số nhân lr gốc; mỗi hệ số chạy cả ga@s và scrub@s
LR_FTYPES     = "random-450"           # thêm loại F khác nếu muốn, vd "random-450 random-1350 hard-450"
LR_PLATEAU_W  = 10                     # dừng khi tới vùng loss ổn định (TB nửa sau vs nửa trước của 10 epoch đổi < 5%); None = tắt
LR_TAG        = "lrgrid"

scales = " ".join(str(s) for s in LR_SCALES)
plat = f"--plateau_w {LR_PLATEAU_W}" if LR_PLATEAU_W else ""
for s in SEEDS:
    run(cli("traj", [s], f"--ftypes {LR_FTYPES} --lr_scales {scales} --tag {LR_TAG} {plat}"))
    run(f'cd /kaggle/working && rm -f pre_out_light.zip && zip -qr pre_out_light.zip {os.path.basename(OUT)} -x "{os.path.basename(OUT)}/ckpt/*"', check=False)

In [ ]:
run(cli("analyze", SEEDS, f"--ftypes {LR_FTYPES} --lr_scales {scales} --tag {LR_TAG}"))

# Dừng vì lý do gì (không có trần epoch): all_rules_decided / plateau / diverged
for m in sorted(glob.glob(OUT + "/logs/traj_*@*_meta.json")):
    d = json.load(open(m, encoding="utf-8"))
    print(d["run_id"], "| epoch", d["n_epochs"], "| dừng:", d["stop_reason"], "| luật chưa quyết định:", d["n_rules_undecided"], "| diverged", d["diverged"])

# Mục 2b của báo cáo lưới lr: L/A trên F, R, V, test của checkpoint do luật chọn, so với RETRAIN, đủ s = 1 và các s nhỏ hơn
cmp_ = pd.read_csv(f"{OUT}/results_{LR_TAG}/compare_retrain.csv")
print(cmp_.to_string(index=False))
print("\nBáo cáo đầy đủ: ", f"{OUT}/results_{LR_TAG}/report_auto.md")

## 4. Đóng gói
`pre_out_light.zip` (không kèm checkpoint nặng) nằm ở `/kaggle/working`; tải về hoặc dùng làm `RESTORE_FROM`. Gửi kèm `results/gate1.csv`, `results/compare_retrain.csv`, `results/report_auto.md`, và các dòng `stage1_run` ở trên.

In [ ]:
run(f'cd /kaggle/working && rm -f pre_out_light.zip && zip -qr pre_out_light.zip {os.path.basename(OUT)} -x "{os.path.basename(OUT)}/ckpt/*" && ls -la pre_out_light.zip')